# SYNERA Cryptographic Provenance Ledger Demo

**Trust Backbone & Tamper-Evident Verification (Part 2)**  
This notebook demonstrates the cryptographic provenance layer of SYNERA:
1. Registration of a dataset with canonical SHA-256 hashing and Ed25519 digital signature.
2. Merkle tree computation for artifact validation.
3. Hash-chained append-only execution logging (`EXPERIMENT_RUN`).
4. Detection of 1-byte file corruption by `IntegrityGuard` and automated `INTEGRITY_ALERT` block emission.
5. Full cryptographic chain verification with `Ledger.verify()`.
6. Visualizing the immutable hash chain using Matplotlib.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import patches

from synera_core.config import find_repo_root
from synera_core.provenance import (
    Ledger,
    register_file,
)
from synera_core.security.integrity import IntegrityGuard

# Set up paths relative to repository root
repo_root = find_repo_root()
demo_dir = repo_root / "artifacts" / "demo_ledger"
demo_dir.mkdir(parents=True, exist_ok=True)
db_path = demo_dir / "provenance.db"
key_path = demo_dir / "signer.key"

if db_path.exists():
    db_path.unlink()
if key_path.exists():
    key_path.unlink()

ledger = Ledger(db_path, key_path)
guard = IntegrityGuard(ledger)

print(f"Initialized Ledger at: {db_path}")
print(f"Signer Ed25519 Public Key: {ledger.signer[:16]}...{ledger.signer[-8:]}")

Initialized Ledger at: D:\synera1.0\synera\artifacts\demo_ledger\provenance.db
Signer Ed25519 Public Key: 4b54409cca2029fa...9a7934a4


In [2]:
# Step 1: Register dummy scientific dataset
data_path = demo_dir / "simulated_multi_stressor.csv"
df = pd.DataFrame({
    "radiation_gy": [0.0, 0.5, 1.0, 2.0],
    "temperature_c": [22.0, 28.0, 32.0, 37.0],
    "gene_expr_fold": [1.0, 1.45, 2.10, 4.35],
})
df.to_csv(data_path, index=False)

dataset_hash, block0 = register_file(
    ledger=ledger,
    path=data_path,
    kind="dataset",
    meta={"provenance": "SIMULATED", "source": "NASA OSDR Benchmark", "samples": len(df)},
    manifest_path=demo_dir / "MANIFEST.json",
)

print(f"Dataset SHA-256: {dataset_hash}")
print(f"Registered in Block #{block0.index} [{block0.block_type}]")
print(f"Block Hash: {block0.block_hash}")
print(f"Ed25519 Signature: {block0.signature[:16]}...")

Dataset SHA-256: f26b269139af3857bdfd15fffc378cc525649b15f8f86777857fd0b7d837ef30
Registered in Block #0 [DATASET_REGISTERED]
Block Hash: f1d73a0d018429cd4bba96cab472a4007938181c44f34d97c10a6b81db7cb159
Ed25519 Signature: 1dfb29e5f3ece51f...


In [3]:
# Step 2: Record an experiment run block referencing the dataset artifact
block1 = ledger.append(
    block_type="EXPERIMENT_RUN",
    payload={
        "experiment_id": "EXP-MULTISTRESSOR-001",
        "model": "gbm_interaction_detector",
        "seed": 42,
        "metrics": {"synergy_fdr": 0.012, "detected_pairs": 3},
    },
    artifact_hashes=[dataset_hash],
)

print(f"Experiment Block #{block1.index} [{block1.block_type}] appended.")
print(f"Block Hash:    {block1.block_hash}")
print(f"Previous Hash: {block1.prev_hash}")
print(f"Merkle Root:   {block1.merkle_root}")

Experiment Block #1 [EXPERIMENT_RUN] appended.
Block Hash:    bbbecf027ec120ce39375434fffdb1cfd05a13151e8a3146b6982f4ee48287d1
Previous Hash: f1d73a0d018429cd4bba96cab472a4007938181c44f34d97c10a6b81db7cb159
Merkle Root:   78638e93c9e382c91598f80e37a5fc80f87811f6a71b3218f2deeb3cbf1c6951


In [4]:
# Step 3: Simulate 1-byte tampering in a copy and audit with IntegrityGuard
corrupt_path = demo_dir / "corrupted_dataset.csv"
raw_bytes = bytearray(data_path.read_bytes())
# Flip one bit in byte position 30
raw_bytes[30] ^= 0x01
corrupt_path.write_bytes(raw_bytes)

print("1. Auditing original clean dataset:")
res_clean = guard.check_dataset(data_path)
print(f"   Status: {'PASS' if res_clean.ok else 'FAIL'} | Detail: {res_clean.detail}")

print("\n2. Auditing mutated dataset against expected hash:")
res_tampered = guard.check_dataset(corrupt_path, expected_hash=dataset_hash)
print(f"   Status: {'PASS' if res_tampered.ok else 'FAIL'}")
print(f"   Detail: {res_tampered.detail}")
print(f"   Expected Hash: {res_tampered.expected_hash}")
print(f"   Actual Hash:   {res_tampered.actual_hash}")

# Note: IntegrityGuard automatically emitted an INTEGRITY_ALERT block
latest_block = ledger.head()
print("\n3. Automated Ledger Alert Generated:")
print(f"   Block #{latest_block.index}: [{latest_block.block_type}]")
print(f"   Alert Detail: {latest_block.payload.get('detail')}")

1. Auditing original clean dataset:
   Status: PASS | Detail: OK

2. Auditing mutated dataset against expected hash:
   Status: FAIL
   Detail: Hash mismatch for D:/synera1.0/synera/artifacts/demo_ledger/corrupted_dataset.csv: expected f26b269139af3857bdfd15fffc378cc525649b15f8f86777857fd0b7d837ef30, got 134cf767696cdf146f68a3838aceb02449d361df71a898e96d7bae4ea30f401c
   Expected Hash: f26b269139af3857bdfd15fffc378cc525649b15f8f86777857fd0b7d837ef30
   Actual Hash:   134cf767696cdf146f68a3838aceb02449d361df71a898e96d7bae4ea30f401c

3. Automated Ledger Alert Generated:
   Block #2: [INTEGRITY_ALERT]
   Alert Detail: Hash mismatch for D:/synera1.0/synera/artifacts/demo_ledger/corrupted_dataset.csv: expected f26b269139af3857bdfd15fffc378cc525649b15f8f86777857fd0b7d837ef30, got 134cf767696cdf146f68a3838aceb02449d361df71a898e96d7bae4ea30f401c


In [5]:
# Step 4: Verify ledger chain integrity
report = ledger.verify()
print("=" * 60)
print("SYNERA LEDGER VERIFICATION REPORT")
print(f"  Valid:         {report['valid']}")
print(f"  Total Height:  {report['height']} blocks")
print(f"  Head Hash:     {report['head']}")
print(f"  Issues Count:  {len(report['issues'])}")
print("=" * 60)

SYNERA LEDGER VERIFICATION REPORT
  Valid:         True
  Total Height:  3 blocks
  Head Hash:     1e1b4332b1455e17c549090e9fad52a682cc607316b220cc8228206becb03b49
  Issues Count:  0


In [6]:
# Step 5: Plot evidence chain with matplotlib
blocks = list(ledger.iter_blocks())
fig, ax = plt.subplots(figsize=(11, 3.5), dpi=150)

box_width, box_height = 2.4, 1.2
spacing = 3.2

colors = {
    "GENESIS": "#1e3d59",
    "DATASET_REGISTERED": "#17b978",
    "EXPERIMENT_RUN": "#438a5e",
    "INTEGRITY_ALERT": "#d7263d",
}

for i, b in enumerate(blocks):
    x = i * spacing
    y = 1.0
    c = colors.get(b.block_type, "#333333")

    rect = patches.FancyBboxPatch(
        (x, y), box_width, box_height,
        boxstyle="round,pad=0.08",
        facecolor=c, edgecolor="#111111", alpha=0.92, lw=1.2
    )
    ax.add_patch(rect)

    ax.text(x + box_width / 2, y + 0.92, f"Block #{b.index}", ha="center", va="center", color="white", fontweight="bold", fontsize=9)
    ax.text(x + box_width / 2, y + 0.65, f"{b.block_type}", ha="center", va="center", color="#f0f0f0", fontsize=7.5, fontweight="semibold")
    ax.text(x + box_width / 2, y + 0.38, f"Hash: {b.block_hash[:8]}...", ha="center", va="center", color="#e0e0e0", fontsize=6.5, fontfamily="monospace")
    ax.text(x + box_width / 2, y + 0.16, f"Prev: {b.prev_hash[:8]}...", ha="center", va="center", color="#cccccc", fontsize=6.5, fontfamily="monospace")

    if i < len(blocks) - 1:
        ax.annotate(
            "",
            xy=((i + 1) * spacing, y + box_height / 2),
            xytext=(x + box_width, y + box_height / 2),
            arrowprops={"arrowstyle": "->", "color": "#444444", "lw": 1.8, "mutation_scale": 12},
        )

ax.set_xlim(-0.4, len(blocks) * spacing)
ax.set_ylim(0.4, 2.6)
ax.axis("off")
plt.title("SYNERA Cryptographic Evidence Chain (Ed25519 Signed SQLite Ledger)", fontsize=11, fontweight="bold", pad=12)
plt.tight_layout()
plot_file = demo_dir / "chain_plot.png"
plt.savefig(plot_file, dpi=150)
print(f"Saved chain plot visualization to {plot_file}")
plt.close()

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved chain plot visualization to D:\synera1.0\synera\artifacts\demo_ledger\chain_plot.png
